# Fair comparison: MLP+Bspline vs KAN on function fitting

## What this notebook does

Reproduces the pykan example notebook (`Example_1_function_fitting.ipynb`)
that fits f(x, y) = exp(sin(πx) + y²) with a KAN, but with the model replaced
by an MLP that uses learnable B-spline activations instead of fixed ones
(ReLU/GELU). The point is to test the claim from Yu et al. (2024,
"KAN or MLP: A Fairer Comparison") that on symbolic-formula regression,
KAN's advantage comes from the B-spline activation itself, not from KAN's
edge-based architecture — and therefore an MLP equipped with B-spline
activation should match KAN under fair comparison.

This claim is tested in the paper's Section 5.2 (Architecture Ablation),
Figure 14. The paper uses MLP hidden widths of 10 or 20 on symbolic formula
tasks with higher-dimensional inputs, where param-matching naturally allows
a larger h. This notebook tests the same claim on a 2D input function, which
forces h=3 under strict param-matching — a stricter regime than the paper
itself used. See "Connection to paper's regime" below.

We test this on one specific function. One function is not a benchmark.
Whatever we find is one data point about the paper's claim, not a verdict.

## The setup

**Model under test (MLP+Bspline):**

`Linear(2 → h) → BSplineActivation(h) → Linear(h → 1)`

- Two ordinary linear layers with a learnable B-spline activation in the
  middle. One spline per hidden neuron, h splines total.
- B-spline activation has only spline coefficients — no SiLU shortcut,
  no scaling factors. This is deliberate: we're testing "spline activation
  alone", as the paper's ablation does.
- Spline coefficients initialised from a small random normal (std 0.1).
  Zero init was tested first and shown to hurt; small random is the
  standard neutral choice and matches what KAN does internally.

**Reference model (KAN), untouched:**

`KAN(width=[2, 1, 1], grid=G, k=3)` from pykan, run in a separate notebook
on the same dataset.

## Fairness: what we controlled

| Variable | Control |
|---|---|
| Dataset | `create_dataset(f, n_var=2, train_num=1000)`, same for both |
| Optimiser | LBFGS (lr=1, max_iter=20, strong_wolfe line search) |
| Steps | 200 LBFGS steps per grid size, same sweep |
| Grid sweep | G ∈ {3, 5, 10, 20, 50, 100}, with `refine` between steps |
| Spline order | k=3 (cubic) |
| Grid range | [-1, 1] |
| Spline coef init | small random normal (matches KAN's behaviour) |
| Regularisation | both run with effective λ=0 (KAN's default; verified in pykan source) |
| **Parameter count** | h chosen at each G to match KAN's params (see below) |

## How we chose h (hidden width of MLP+Bspline)

We derive parameter counts from first principles with k=3 (cubic spline).

**KAN [2, 1, 1]:** There are 3 edges total (2 input→1 hidden, 1 hidden→1 output).
Each edge has (G + k) = (G + 3) spline coefficients, 1 spline weight, and
1 shortcut weight. Plus 2 bias terms (one per non-input node).

```
P_KAN = 3 × (G + 3 + 1 + 1) + 2 = 3(G + 5) + 2 = 3G + 17
```

Wait — using the paper's formula directly (Section 3): `(d_in × d_out) × (G + K + 3) + d_out`
per layer. For KAN [2,1,1] with two layers (2→1) and (1→1):

```
Layer 1 (2→1): (2×1) × (G+3+3) + 1 = 2(G+6) + 1 = 2G + 13
Layer 2 (1→1): (1×1) × (G+3+3) + 1 = (G+6) + 1 = G + 7
P_KAN = (2G+13) + (G+7) = 3G + 20
```

**MLP+Bspline [2, h, 1]:**

```
Linear(2→h):          2h weights + h biases  = 3h
BSplineActivation(h): h × (G+k) coefficients = h(G+3)
Linear(h→1):          h weights + 1 bias     = h + 1
P_MLP = 3h + h(G+3) + h + 1 = h(G+7) + 1
```

Setting equal and solving for h:

```
h(G+7) + 1 = 3G + 20  →  h = round((3G + 19) / (G + 7))
```

For all G in our sweep, this gives **h = 3**. The MLP+Bspline width is
therefore fixed at [2, 3, 1] throughout the sweep — only the grid
resolution G changes, mirroring how the KAN keeps width [2,1,1] fixed
and only refines its grid.

Param-count mismatch from integer rounding: +1% to +7%, with MLP+Bspline
having slightly more params than KAN. Documented but small.

## Connection to paper's regime

The paper (Section 5.2, Fig. 14) tests MLP+BSpline on 8 special functions
from `scipy.special`, which have multiple inputs. With higher input dimensions,
param-matching against larger KAN architectures naturally produces h = 10–20,
giving the splines a richer representation to work with.

Here, the 2D input combined with the small KAN [2,1,1] collapses param-matched
h to 3. This means the Linear(2→3) projection is the only thing the splines
ever see — they never touch the raw inputs. KAN [2,1,1] by contrast places
splines directly on the 2 inputs with no projection in front of them.

In other words: **this notebook tests a stricter version of the paper's claim
than the paper itself tested.** The paper's result (MLP+BSpline matches KAN)
likely holds in the regime it was measured (higher-dim inputs, larger h). Our
result (MLP+BSpline fails to match KAN) identifies where that claim breaks down.

## What we did NOT control (and why)

1. **Where splines live.** KAN: on edges, acting directly on inputs.
   MLP+Bspline: on hidden neurons, downstream of a linear projection.
   This is the architectural difference under test — controlling it
   would defeat the purpose.

2. **SiLU shortcut branch.** KAN has it; our MLP+Bspline doesn't.
   Deliberate, per the paper's ablation: we test the spline activation
   in isolation. Adding the shortcut would change the question to
   "KAN with reshuffled layer order."

3. **Wall-clock time and FLOPs.** Implementation-dependent and not an
   architecture property. FLOPs could be reported as a second plot
   axis later; not done yet.

## What we tried that didn't work

### Attempt 1: zero coefficient init (the original code)

The starting code initialised spline coefficients to zero. At step 0, the
spline outputs zero everywhere; the entire signal comes from the linear
layers. LBFGS has to bootstrap nonlinearity from scratch.

Switched to small random normal init (std 0.1). G=3 result improved from
2.14e-01 to 1.60e-01 (~25% lower RMSE). This is the version we kept.

### Attempt 2: periodic grid updates during training

KAN's default `fit()` updates spline grids 10 times across the first 50
training steps, then freezes the grid. Our `fit()` only updated once at
the start. Added the same schedule to MLP+Bspline (with curve2coef shape
preservation and LBFGS reset after each update).

Result: worse, not better. Test RMSE at G=100 went from 1.22e-01 to
2.60e-01. Most likely reason: in KAN, splines see a stable input
distribution (the dataset itself). In MLP+Bspline, splines see
pre-activations that shift as the linear layers learn. Re-adapting the
grid mid-training destabilises convergence here. Grid updates appear to
be architecturally tied to where splines are placed.

Reverted to single grid update at start of `fit`.

## What we ended up with

The canonical run (Run 1 / reverted state) gives:

| G | params | train RMSE | test RMSE |
|---|---|---|---|
| 3 | 31 | 1.71e-01 | 1.85e-01 |
| 5 | 37 | 1.06e-01 | 1.10e-01 |
| 10 | 52 | 1.01e-01 | 1.04e-01 |
| 20 | 82 | 9.86e-02 | 1.03e-01 |
| 50 | 172 | 9.37e-02 | 1.07e-01 |
| 100 | 322 | 8.41e-02 | 1.22e-01 |

KAN on the same dataset (from separate run):

| G | params | train RMSE | test RMSE |
|---|---|---|---|
| 3 | 29 | 1.35e-02 | 1.39e-02 |
| 5 | 35 | 7.28e-03 | 7.23e-03 |
| 10 | 50 | 4.81e-04 | 5.13e-04 |
| 20 | 80 | 4.61e-05 | 6.12e-05 |
| 50 | 170 | 1.54e-05 | 3.12e-05 |
| 100 | 320 | 1.21e-05 | 3.48e-05 |

## What we observed

1. **MLP+Bspline plateaus around test RMSE ≈ 1e-1.** Train RMSE drops
   from 1.71e-01 to 8.4e-02 across the sweep — a 2× improvement using
   10× more parameters. KAN drops ~400× over the same sweep.

2. **MLP+Bspline overfits from G=10 onward.** Train RMSE keeps decreasing
   while test RMSE drifts upward. The splines memorise training noise
   that the small linear-projection bottleneck can't generalise.

3. **The gap grows with G**, from 13× at G=3 to 3500× at G=100. More
   capacity helps KAN dramatically and helps MLP+Bspline almost not at
   all.

## What we can claim, and what we can't

**Can claim:** On this specific 2D function, with strict param matching
forcing h=3, the paper's claim does not hold. The B-spline activation
alone is not enough — the placement of splines (directly on raw inputs
vs downstream of a linear projection) matters substantially. This
identifies the boundary condition under which the paper's result breaks.

**Cannot claim:** That the paper is wrong in general. One function, one
regime. The paper tested 8 special functions on higher-dim inputs where
h is naturally larger and the bottleneck argument weakens. Our finding
is consistent with the paper's broader results holding in that regime.

**Likely explanation for the gap on this specific function:** the input
dimension is only 2. Param-matching forces h=3, which means the
Linear(2→3) bottleneck projects 2D input into 3D before any spline ever
sees it. Whatever 3 channels that linear projection produces is what the
splines have to work with. By contrast, KAN's [2,1,1] puts splines
directly on the inputs — no projection. On higher-dim inputs, h would
be larger, and the bottleneck argument weakens.

## Open questions for follow-up (ranked by diagnostic value)

1. **FLOPs matching (most diagnostic).** KAN's De Boor-Cox computation
   is much more expensive per parameter. Under FLOP matching, MLP+Bspline
   gets a larger budget → larger h, on the same 2D function and dataset.
   This directly tests whether the bottleneck is the cause, without
   changing the problem.

2. **Higher-dim functions.** Testing on functions with ≥4 inputs where
   param-matching naturally gives h=10–20 would replicate the paper's
   actual regime and confirm whether the gap closes there.

3. **SiLU shortcut in BSplineActivation.** Adding KAN's shortcut branch
   tests a different hypothesis: "spline + shortcut on neurons" vs
   "spline on edges." Likely a smaller effect than the bottleneck, but
   worth isolating.

In [1]:
from kan import *
from kan.spline import B_batch, coef2curve, curve2coef, extend_grid
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# Build the dataset once, with a fixed seed, so KAN and MLP+Bspline see identical data.
torch.manual_seed(42)
f = lambda x: torch.exp(torch.sin(torch.pi * x[:, [0]]) + x[:, [1]]**2)
dataset = create_dataset(f, n_var=2, train_num=1000, device=device)

print("train input :", dataset['train_input'].shape)
print("train label :", dataset['train_label'].shape)
print("test input  :", dataset['test_input'].shape)
print("test label  :", dataset['test_label'].shape)

Device: cpu
train input : torch.Size([1000, 2])
train label : torch.Size([1000, 1])
test input  : torch.Size([1000, 2])
test label  : torch.Size([1000, 1])


In [2]:
class BSplineActivation(nn.Module):
    def __init__(self, in_features, grid=3, k=3):
        super().__init__()
        self.in_features = in_features
        self.grid_size = grid
        self.k = k

        h = 2.0 / grid
        grid_pts = torch.linspace(-1 - k*h, 1 + k*h, grid + 2*k + 1)
        self.register_buffer('grid', grid_pts.unsqueeze(0).expand(in_features, -1).clone())
        self.coef = nn.Parameter(torch.randn(in_features, grid + k) * 0.1)

    def forward(self, x):
        basis = B_batch(x, self.grid, k=self.k)
        return torch.einsum('bik,ik->bi', basis, self.coef)

    def update_grid_from_samples(self, x):
        with torch.no_grad():
            num = self.grid_size
            k = self.k

            x_sorted, _ = torch.sort(x, dim=0)
            ids = torch.linspace(0, x.shape[0] - 1, num + 1).long()
            grid_adaptive = x_sorted[ids, :].T

            margin = 0.01
            h = (grid_adaptive[:, [-1]] - grid_adaptive[:, [0]] + 2 * margin) / num
            grid_uniform = grid_adaptive[:, [0]] - margin + h * torch.arange(num + 1).to(x.device)

            grid_eps = 0.02
            grid_base = grid_eps * grid_uniform + (1 - grid_eps) * grid_adaptive
            grid_extended = extend_grid(grid_base, k_extend=k)
            self.register_buffer('grid', grid_extended.to(x.device))


class BSplineMLP(nn.Module):
    def __init__(self, width, grid=3, k=3, seed=0, device='cpu'):
        super().__init__()
        torch.manual_seed(seed)
        self.width = width
        self.depth = len(width) - 1
        self.k = k
        self.device = device
        self.cache_data = None

        linears = []
        activations = []
        for i in range(self.depth - 1):
            linears.append(nn.Linear(width[i], width[i+1]))
            activations.append(BSplineActivation(width[i+1], grid=grid, k=k))
        linears.append(nn.Linear(width[-2], width[-1]))

        self.linears = nn.ModuleList(linears)
        self.activations = nn.ModuleList(activations)
        self.to(device)

    def forward(self, x):
        for i in range(self.depth - 1):
            x = self.linears[i](x)
            x = self.activations[i](x)
        x = self.linears[-1](x)
        return x

    def update_grids(self, x):
        with torch.no_grad():
            for i in range(self.depth - 1):
                x = self.linears[i](x)
                self.activations[i].update_grid_from_samples(x)
                x = self.activations[i](x)

    def refine(self, new_grid):
        with torch.no_grad():
            x = self.cache_data
            for i in range(self.depth - 1):
                x = self.linears[i](x)
                pre_act = x.clone()

                act = self.activations[i]
                k = act.k
                num = new_grid

                x_sorted, _ = torch.sort(pre_act, dim=0)
                ids = torch.linspace(0, pre_act.shape[0] - 1, num + 1).long()
                grid_adaptive = x_sorted[ids, :].T

                margin = 0.01
                h = (grid_adaptive[:, [-1]] - grid_adaptive[:, [0]] + 2 * margin) / num
                grid_uniform = grid_adaptive[:, [0]] - margin + h * torch.arange(num + 1).to(self.device)

                grid_eps = 0.02
                grid_base = grid_eps * grid_uniform + (1 - grid_eps) * grid_adaptive
                new_grid_buf = extend_grid(grid_base, k_extend=k).to(self.device)

                y_current = act(pre_act)
                y_3d = y_current.unsqueeze(2)
                new_coef = curve2coef(pre_act, y_3d, new_grid_buf, k)

                act.register_buffer('grid', new_grid_buf)
                act.coef = nn.Parameter(new_coef[:, 0, :])
                act.grid_size = new_grid

                x = act(pre_act)
        return self

    def fit(self, dataset, opt="LBFGS", steps=100, lr=1.):
        self.cache_data = dataset['train_input'].to(self.device)
        self.update_grids(self.cache_data)

        optimizer = torch.optim.LBFGS(self.parameters(), lr=lr,
                                       max_iter=20,
                                       line_search_fn="strong_wolfe")
        loss_fn = lambda pred, y: torch.mean((pred - y) ** 2)
        results = {'train_loss': [], 'test_loss': []}
        pbar = tqdm(range(steps), desc='description', ncols=100)

        train_input = dataset['train_input'].to(self.device)
        train_label = dataset['train_label'].to(self.device)
        test_input  = dataset['test_input'].to(self.device)
        test_label  = dataset['test_label'].to(self.device)

        def closure():
            optimizer.zero_grad()
            loss = loss_fn(self.forward(train_input), train_label)
            loss.backward()
            return loss

        for _ in pbar:
            optimizer.step(closure)
            with torch.no_grad():
                train_loss = torch.sqrt(loss_fn(self.forward(train_input), train_label))
                test_loss  = torch.sqrt(loss_fn(self.forward(test_input),  test_label))
            results['train_loss'].append(train_loss.item())
            results['test_loss'].append(test_loss.item())
            pbar.set_description("| train_loss: %.2e | test_loss: %.2e |" % (train_loss, test_loss))

        return results

In [3]:
model = BSplineMLP(width=[2, 3, 1], grid=3, k=3, seed=1, device=device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: BSplineMLP(width=[2, 3, 1], grid=3, k=3)")
print(f"Total parameters: {n_params}")
print(f"Expected from formula h*(G+7)+1 = 3*(3+7)+1 = 31")

Model: BSplineMLP(width=[2, 3, 1], grid=3, k=3)
Total parameters: 31
Expected from formula h*(G+7)+1 = 3*(3+7)+1 = 31


In [4]:
results_g3 = model.fit(dataset, opt="LBFGS", steps=20)

print(f"\nFinal train RMSE: {results_g3['train_loss'][-1]:.4e}")
print(f"Final test  RMSE: {results_g3['test_loss'][-1]:.4e}")

| train_loss: 1.60e-01 | test_loss: 1.69e-01 |: 100%|███████████████| 20/20 [00:00<00:00, 37.38it/s]


Final train RMSE: 1.6018e-01
Final test  RMSE: 1.6864e-01


In [5]:
model = model.refine(10)

n_params = sum(p.numel() for p in model.parameters())
print(f"After refine to G=10:")
print(f"Total parameters: {n_params}")
print(f"Expected from formula h*(G+7)+1 = 3*(10+7)+1 = 52")

After refine to G=10:
Total parameters: 52
Expected from formula h*(G+7)+1 = 3*(10+7)+1 = 52


In [6]:
results_g10 = model.fit(dataset, opt="LBFGS", steps=20)

print(f"\nFinal train RMSE: {results_g10['train_loss'][-1]:.4e}")
print(f"Final test  RMSE: {results_g10['test_loss'][-1]:.4e}")
print(f"\nFor comparison, G=3 was: train {1.60e-01:.4e}, test {1.69e-01:.4e}")
print(f"For comparison, KAN at G=10 gets: train ~4.18e-03 (per pykan notebook)")

| train_loss: 1.14e-01 | test_loss: 1.16e-01 |: 100%|███████████████| 20/20 [00:00<00:00, 45.69it/s]


Final train RMSE: 1.1380e-01
Final test  RMSE: 1.1630e-01

For comparison, G=3 was: train 1.6000e-01, test 1.6900e-01
For comparison, KAN at G=10 gets: train ~4.18e-03 (per pykan notebook)


In [7]:
grids = np.array([3, 5, 10, 20, 50, 100])

mlp_train_losses = []
mlp_test_losses  = []
mlp_params_per_grid = []
mlp_final_test_rmse_per_grid = []
mlp_final_train_rmse_per_grid = []

steps = 200
k = 3

for i in range(len(grids)):
    if i == 0:
        # Fresh model at the first grid size, with the matched hidden width h=3
        model_mlp = BSplineMLP(width=[2, 3, 1], grid=grids[i], k=k, seed=0, device=device)
    else:
        # Refine: increase grid resolution, preserving learned spline shapes
        model_mlp = model_mlp.refine(grids[i])

    n_params = sum(p.numel() for p in model_mlp.parameters())
    mlp_params_per_grid.append(n_params)
    print(f"\n--- Training at G={grids[i]} (params={n_params}) ---")

    results = model_mlp.fit(dataset, opt="LBFGS", steps=steps)

    mlp_train_losses += results['train_loss']
    mlp_test_losses  += results['test_loss']
    mlp_final_train_rmse_per_grid.append(results['train_loss'][-1])
    mlp_final_test_rmse_per_grid.append(results['test_loss'][-1])

print("\n=== Summary ===")
print(f"{'G':>4} {'params':>8} {'final train RMSE':>18} {'final test RMSE':>18}")
for i, G in enumerate(grids):
    print(f"{G:>4} {mlp_params_per_grid[i]:>8} {mlp_final_train_rmse_per_grid[i]:>18.4e} {mlp_final_test_rmse_per_grid[i]:>18.4e}")


--- Training at G=3 (params=31) ---


| train_loss: 1.71e-01 | test_loss: 1.85e-01 |: 100%|█████████████| 200/200 [00:02<00:00, 99.58it/s]



--- Training at G=5 (params=37) ---


| train_loss: 1.06e-01 | test_loss: 1.10e-01 |: 100%|████████████| 200/200 [00:01<00:00, 142.84it/s]



--- Training at G=10 (params=52) ---


...


--- Training at G=50 (params=172) ---


| train_loss: 9.37e-02 | test_loss: 1.07e-01 |: 100%|█████████████| 200/200 [00:03<00:00, 58.15it/s]



--- Training at G=100 (params=322) ---


| train_loss: 8.41e-02 | test_loss: 1.22e-01 |: 100%|█████████████| 200/200 [00:04<00:00, 48.57it/s]


=== Summary ===
   G   params   final train RMSE    final test RMSE
   3       31         1.7113e-01         1.8511e-01
   5       37         1.0589e-01         1.1029e-01
  10       52         1.0052e-01         1.0362e-01
  20       82         9.8616e-02         1.0346e-01
  50      172         9.3749e-02         1.0683e-01
 100      322         8.4078e-02         1.2151e-01


In [8]:
import inspect
print("'make_optimizer' in fit source:", 'make_optimizer' in inspect.getsource(BSplineMLP.fit))
print("\n--- first 30 lines of BSplineMLP.fit ---")
print('\n'.join(inspect.getsource(BSplineMLP.fit).split('\n')[:30]))

'make_optimizer' in fit source: False

--- first 30 lines of BSplineMLP.fit ---
    def fit(self, dataset, opt="LBFGS", steps=100, lr=1.):
        self.cache_data = dataset['train_input'].to(self.device)
        self.update_grids(self.cache_data)

        optimizer = torch.optim.LBFGS(self.parameters(), lr=lr,
                                       max_iter=20,
                                       line_search_fn="strong_wolfe")
        loss_fn = lambda pred, y: torch.mean((pred - y) ** 2)
        results = {'train_loss': [], 'test_loss': []}
        pbar = tqdm(range(steps), desc='description', ncols=100)

        train_input = dataset['train_input'].to(self.device)
        train_label = dataset['train_label'].to(self.device)
        test_input  = dataset['test_input'].to(self.device)
        test_label  = dataset['test_label'].to(self.device)

        def closure():
            optimizer.zero_grad()
            loss = loss_fn(self.forward(train_input), train_label)
            loss.